# Kiểm tra riding → ride hay rid (CPU)

**Accelerator None, Internet ON. Không cần Add Input, Dataset hay API token.**
40captionTRAIN có chủ đích +5câu kiểm soát ngôn ngữ; không chạy lại toànTRAIN.
Cùng spaCy3.8.11/Thinc8.3.11/NumPy2.1.3/model3.8.0 và so khớp dữ liệu model
với Output126. Xuất nguyên từ, POS, tag, lemma và bảng HTML; không sửa bank/ROI.

Save & Run All. Gửi `csa_lemma_trace_check_review.zip`.
Không chạy model caption, GPU, CLIP text, huấn luyện hoặc lưu Dataset.
Dữ liệu Output126 giữ nguyên; có thể lưu riêng bằng notebook SAVE126 đã gửi.


In [ ]:
import hashlib, json, os, queue, signal, subprocess, sys, tempfile, threading, time, traceback, urllib.request, zipfile
from pathlib import Path
OUTPUT = Path('/kaggle/working/csa_lemma_trace_check')
if OUTPUT.exists(): raise RuntimeError('Preserve previous ZIP and use a fresh CPU session.')
OUTPUT.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix='csa-lemma-trace-'))
LOG = OUTPUT/'notebook.log'
MODULES={'lemma_trace.py': '"""Pure token-record inspection, no model imports and no token/lemma mutation."""\nfrom collections import Counter\n\nEXPECTED = dict(riding=\'ride\', rides=\'ride\', ride=\'ride\', ridden=\'ride\', rode=\'ride\', rid=\'rid\', ridding=\'rid\')\n\n\ndef inspect_doc(doc):\n    tokens, focuses = [], []\n    for index, token in enumerate(doc):\n        record = dict(index=index, surface=token.text, pos=token.pos_, tag=token.tag_,\n                      morph=str(token.morph), lemma=token.lemma_)\n        tokens.append(record)\n        expected = EXPECTED.get(token.text.lower())\n        if expected is not None:\n            status = \'NOT_VERB_REVIEW\' if token.pos_ != \'VERB\' else \'MATCH\' if token.lemma_.lower() == expected else \'MISMATCH\'\n            focuses.append(dict(record, expected_lemma_for_verb=expected, status=status))\n    return dict(tokens=tokens, focus=focuses)\n\n\ndef summarize(records):\n    counts = Counter()\n    transitions = Counter()\n    for record in records:\n        for focus in record[\'focus\']:\n            counts[(record[\'kind\'], focus[\'status\'])] += 1\n            transitions[(record[\'kind\'], focus[\'surface\'].lower(), focus[\'pos\'], focus[\'lemma\'])] += 1\n    return dict(case_count=len(records), focus_counts=[dict(kind=kind, status=status, count=count)\n            for (kind, status), count in sorted(counts.items())],\n        transitions=[dict(kind=kind, surface=surface, pos=pos, lemma=lemma, count=count)\n            for (kind, surface, pos, lemma), count in sorted(transitions.items())],\n        bank_modified=False, dataset_error_rate_estimated=False)\n', 'run_lemma_trace.py': '"""USER-RUN CPU: reproduce exact POS settings on the frozen small packet."""\nimport argparse\nimport hashlib\nimport html\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport time\nimport traceback\n\nfrom lemma_trace import inspect_doc, summarize\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef run(out, report):\n    import spacy\n    here = Path(__file__).parent\n    packet = json.loads((here/\'trace_cases.json\').read_text(encoding=\'utf-8\'))\n    original = json.loads((here/\'source_environment.json\').read_text(encoding=\'utf-8\'))\n    versions = {n: importlib.metadata.version(n) for n in (\'spacy\', \'thinc\', \'numpy\', \'en-core-web-sm\')}\n    if versions != {n: original[\'versions\'][n] for n in versions}:\n        raise ValueError(\'POS software versions differ from Output126\')\n    spacy.require_cpu()\n    nlp = spacy.load(\'en_core_web_sm\', exclude=[\'parser\', \'ner\'])\n    if nlp.meta[\'version\'] != \'3.8.0\' or set(nlp.pipe_names) != {\'tok2vec\', \'tagger\', \'attribute_ruler\', \'lemmatizer\'}:\n        raise ValueError(\'POS model/pipeline differs\')\n    observed = {str(p.relative_to(nlp.path)): sha(p) for p in sorted(Path(nlp.path).rglob(\'*\')) if p.is_file()}\n    if observed != original[\'pos_model_files_sha256\']:\n        raise ValueError(\'Installed POS model data hashes differ from Output126\')\n    report.update(versions=versions, model_files_matched=len(observed), model_data_identity_verified=True)\n    records = []\n    cases = packet[\'cases\']\n    docs = nlp.pipe((c[\'caption\'] for c in cases), batch_size=256, n_process=1)\n    for number, (case, doc) in enumerate(zip(cases, docs), 1):\n        record = dict(case, **inspect_doc(doc))\n        records.append(record)\n        print(f\'Case {number}/{len(cases)} [{case["kind"]}]: {case["caption"]}\', flush=True)\n        for focus in record[\'focus\']:\n            print(\' \', focus[\'surface\'], \'→\', focus[\'pos\'], \'/\', focus[\'tag\'], \'→\', focus[\'lemma\'],\n                  \'; expected screen:\', focus[\'expected_lemma_for_verb\'], \';\', focus[\'status\'], flush=True)\n    if len(records) != len(cases) or not all(r[\'focus\'] for r in records):\n        raise ValueError(\'Missing case or focus token in trace\')\n    report.update(summarize(records), status=\'LEMMA_TRACE_COMPLETE\')\n    (out/\'trace_records.json\').write_text(json.dumps(records, ensure_ascii=False, indent=2)+\'\\n\', encoding=\'utf-8\')\n    parts = []\n    for number, record in enumerate(records, 1):\n        rows = []\n        for f in record[\'focus\']:\n            rows.append(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(f[k]))+\'</td>\' for k in\n                (\'surface\', \'pos\', \'tag\', \'lemma\', \'expected_lemma_for_verb\', \'status\'))+\'</tr>\')\n        parts.append(\'<section><h2>\'+str(number)+\' · \'+html.escape(record[\'kind\'])+\'</h2><p>\'+html.escape(record[\'caption\'])+\n            \'</p><table><tr><th>Từ</th><th>POS</th><th>Tag</th><th>Lemma thực tế</th><th>Lemma kiểm tra nếu VERB</th><th>Đối chiếu</th></tr>\'+\'\'.join(rows)+\'</table></section>\')\n    page=\'<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>CPU lemma trace</title><style>body{font:16px/1.5 system-ui;max-width:1100px;margin:30px auto;padding:0 18px;color:#17324d}section{margin:22px 0;padding:16px;background:#f3f6fa;border-radius:10px}table{width:100%;border-collapse:collapse}td,th{text-align:left;padding:8px;border-bottom:1px solid #ccd7e1}</style><h1>Token → POS → lemma</h1><p>Giữ nguyên đầu ra spaCy. Đây là mẫu kiểm tra ngôn ngữ có chủ đích, không phải tỷ lệ lỗi toàn dataset. Không đổi bank hoặc huấn luyện mô hình caption.</p>\'+\'\'.join(parts)+\'</html>\'\n    (out/\'trace_review.html\').write_text(page, encoding=\'utf-8\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(); parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args(); out = Path(args.output); out.mkdir(parents=True, exist_ok=True)\n    report = dict(status=\'INCOMPLETE\', bank_modified=False, roi_modified=False,\n                  caption_model_executed=False, gpu_used=False, dataset_error_rate_estimated=False)\n    start = time.monotonic()\n    try:\n        run(out, report)\n    except BaseException:\n        report[\'error\'] = traceback.format_exc()\n        raise\n    finally:\n        report[\'seconds\'] = time.monotonic()-start\n        (out/\'summary.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n'}
MODULE_HASHES={'lemma_trace.py': '93f8abeeb21eadfa515155240ed60d27eb4f2b966490f7d29f009488ed942500', 'run_lemma_trace.py': '758e887afa613a7f55a835214cde92c02026b3ba889d1631e45b69a859bd5647'}
PACKET='{\n  "schema": "csa_lemma_trace_cases_v1",\n  "source_dataset_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n  "selection": "First up to6 TRAIN examples per surface +3 discussed images if not already selected; purposive, not representative",\n  "cases": [\n    {\n      "kind": "train_caption",\n      "coco_id": 372938,\n      "filename": "COCO_val2014_000000372938.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "A group of people riding on the back of a loaded red pickup truck.",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 384553,\n      "filename": "COCO_val2014_000000384553.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "A man riding an elephant into some water of a creek.",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 384553,\n      "filename": "COCO_val2014_000000384553.jpg",\n      "split": "restval",\n      "sentence_index": 1,\n      "caption": "man riding an elephant into water surrounded by forest",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 384553,\n      "filename": "COCO_val2014_000000384553.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A man riding an elephant in a river.",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 384553,\n      "filename": "COCO_val2014_000000384553.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "A man in a brown shirt rides an elephant into the water.",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 384553,\n      "filename": "COCO_val2014_000000384553.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A man rides an elephant into a river.",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 79841,\n      "filename": "COCO_val2014_000000079841.jpg",\n      "split": "restval",\n      "sentence_index": 1,\n      "caption": "Two shots of men riding bicycles down city streets.",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 79841,\n      "filename": "COCO_val2014_000000079841.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A man riding a bike in front of a tall building.",\n      "selection_forms": [\n        "riding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 122851,\n      "filename": "COCO_val2014_000000122851.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A man attending to several bundles of banana\'s as a motorcycle rides by.",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 296649,\n      "filename": "COCO_val2014_000000296649.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "people ride their motorcycles beside some cars, passing by an empty street with stores and apartment buildings",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 490081,\n      "filename": "COCO_val2014_000000490081.jpg",\n      "split": "restval",\n      "sentence_index": 1,\n      "caption": "Man rides on the back of a wooden cart being pulled by two cows. ",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 242499,\n      "filename": "COCO_val2014_000000242499.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "People ride on elephants to cross the river.",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 260166,\n      "filename": "COCO_val2014_000000260166.jpg",\n      "split": "restval",\n      "sentence_index": 1,\n      "caption": "Several people are taking a ride on elephants.",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 289899,\n      "filename": "COCO_val2014_000000289899.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "A group of people ride in a canal past buildings.",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 122934,\n      "filename": "COCO_val2014_000000122934.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "The tourists take a leasurely ride in the horse-drawn carriage.",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 440329,\n      "filename": "COCO_val2014_000000440329.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "A group of people standing in a parking lot as a man rides a bull.",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 440329,\n      "filename": "COCO_val2014_000000440329.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A man rides a cow through a parking lot.",\n      "selection_forms": [\n        "rides"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 497312,\n      "filename": "COCO_val2014_000000497312.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A woman is watching a girl ride a horse.",\n      "selection_forms": [\n        "ride"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 409944,\n      "filename": "COCO_val2014_000000409944.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A poster of an angry tennis player adorns a graffiti-ridden wall.",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 328354,\n      "filename": "COCO_val2014_000000328354.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "A dilapidated room with a plaster chunked floor is equipped with debris ridden chairs and a tiny vintage refrigerator.",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 46508,\n      "filename": "COCO_val2014_000000046508.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A elephant being ridden by a little boy.",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 162021,\n      "filename": "COCO_val2014_000000162021.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "Three young people ridding skateboards in the city.",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 39860,\n      "filename": "COCO_val2014_000000039860.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A young man in a green outfit ridding a surfboard. ",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 460781,\n      "filename": "COCO_val2014_000000460781.jpg",\n      "split": "restval",\n      "sentence_index": 0,\n      "caption": "A young man ridding a snowboard sliding down a railing in the snow.",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 136285,\n      "filename": "COCO_val2014_000000136285.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A man standing next to a horse being ridden by a young boy. ",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 477791,\n      "filename": "COCO_val2014_000000477791.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A cat ridding in a car looking out the window with a small building in the background.",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 145562,\n      "filename": "COCO_val2014_000000145562.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A small train being ridden by a few people.",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 227772,\n      "filename": "COCO_val2014_000000227772.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "A young man ridding a skateboard down the handrail of some steps as people watch.",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 312078,\n      "filename": "COCO_val2014_000000312078.jpg",\n      "split": "restval",\n      "sentence_index": 2,\n      "caption": "A young person ridding the waves on a surf board.",\n      "selection_forms": [\n        "ridding"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 246746,\n      "filename": "COCO_val2014_000000246746.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "A horse being ridden in the middle of the grass.",\n      "selection_forms": [\n        "ridden"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 107558,\n      "filename": "COCO_val2014_000000107558.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "there is a skateboarder that has rode down a ramp",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 357354,\n      "filename": "COCO_val2014_000000357354.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "An elephant is being rode on the street",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 493772,\n      "filename": "COCO_val2014_000000493772.jpg",\n      "split": "restval",\n      "sentence_index": 3,\n      "caption": "A person holds an umbrella in the middle of a rode. ",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 84004,\n      "filename": "COCO_val2014_000000084004.jpg",\n      "split": "restval",\n      "sentence_index": 4,\n      "caption": "A young female surfer just rode a wave at the beach",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 7726,\n      "filename": "COCO_val2014_000000007726.jpg",\n      "split": "restval",\n      "sentence_index": 1,\n      "caption": "a truck driving down the rode on a rainy day",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 231782,\n      "filename": "COCO_train2014_000000231782.jpg",\n      "split": "train",\n      "sentence_index": 4,\n      "caption": "Cows walking along side a rode with snow on the mountain behind.",\n      "selection_forms": [\n        "rode"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 14683,\n      "filename": "COCO_train2014_000000014683.jpg",\n      "split": "train",\n      "sentence_index": 3,\n      "caption": "Two jockeys rid inning in a pasture with sheep.",\n      "selection_forms": [\n        "rid"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 69167,\n      "filename": "COCO_train2014_000000069167.jpg",\n      "split": "train",\n      "sentence_index": 4,\n      "caption": "A young boy riding bareback on a horse at a fair",\n      "selection_forms": [\n        "visually_discussed_example"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 263974,\n      "filename": "COCO_train2014_000000263974.jpg",\n      "split": "train",\n      "sentence_index": 0,\n      "caption": "a number of people riding bicycles on a street ",\n      "selection_forms": [\n        "visually_discussed_example"\n      ]\n    },\n    {\n      "kind": "train_caption",\n      "coco_id": 198785,\n      "filename": "COCO_train2014_000000198785.jpg",\n      "split": "train",\n      "sentence_index": 4,\n      "caption": "A group of women riding on the back of a motorcycle.",\n      "selection_forms": [\n        "visually_discussed_example"\n      ]\n    },\n    {\n      "kind": "synthetic_control",\n      "caption": "A boy is riding a horse."\n    },\n    {\n      "kind": "synthetic_control",\n      "caption": "A cyclist rides a bicycle."\n    },\n    {\n      "kind": "synthetic_control",\n      "caption": "People ride horses."\n    },\n    {\n      "kind": "synthetic_control",\n      "caption": "They rid the room of smoke."\n    },\n    {\n      "kind": "synthetic_control",\n      "caption": "They are ridding the room of smoke."\n    }\n  ]\n}\n'
ENVIRONMENT='{"pos_model_files_sha256": {"LICENSE": "3933c176979b68bc6d0bcc902c7d6c130f1d127f476f17ba5cdba8d99cfd0012", "LICENSES_SOURCES": "20d9dfacd215112251f15341edd1a47b1f98bf393a212f10f194ffdc7ee98a90", "README.md": "91b0ed1742ea0803034b4d769c4536e88d9d4da7808d833f2a804b6c54aae220", "accuracy.json": "e7572d7f2f3c18e9921471f9d0b3424f8cf7f1a909ae484fa10b780a13f4c676", "attribute_ruler/patterns": "f8b97f7ebef9b8536bba5f0fc1e73af6183563e96e2325bcbb021b8c6dbc6683", "config.cfg": "65c545ca2da5bd4b87b22d4917318605360b37dee6ed6b18516a036f87319f79", "lemmatizer/lookups/lookups.bin": "eb64f40c0f8396d1762730c0ddf4dad2a52d138f5a389f71a1a1d088173b7737", "meta.json": "7456349002fa8cf31111051bd37fdbea67a1b7f7a0a60ce235466f98a6758125", "ner/cfg": "a7172edadafba9f472e9ac0f2660eec04b6405e471be9e20267b79c67288d22d", "ner/model": "cd93e9ca9126c69f1d10a1039ef67c3d9ec1223b44d079856705970f0f19ff4e", "ner/moves": "d82b13473ebd33e0396004dd22f00c71fe3c1251742bf1f7c4554f23fd3b0af4", "parser/cfg": "e2a240ef01f95e58f5dd4c9d6e90d61f597b8d998966570d3009dd81e8879780", "parser/model": "e0098910535a863d430082ece57455d5fa071cd4ca3a0054e80582076c752b1e", "parser/moves": "ec41318c32eb7f9e4d8f050b2e8466ee67954d2d3455c1aec73abf46c82dd67f", "senter/cfg": "2a4db4a729b6469da1f87d71cc7cd3ed2a73360cc85326821e850836c4671103", "senter/model": "b69e7174a0b6307739adec3c8053ac45fdbad7171c4d3d8bfcd9713ecbc01f8c", "tagger/cfg": "d484c3d9391f37bf9c727aa2eb4c477a24cb103020f95e52dc37842818b50118", "tagger/model": "1ec3d93f38cebe172f2b5c89d84be72856ce42c8f1a64f1699f1d98a771f36b7", "tok2vec/cfg": "f8a5a26e3056eb6fb06deeb3dbccfd88ae74900200c98c70b5966bbb7ec9d4de", "tok2vec/model": "e84fc06eb319c94d28e460fc334e292120b01f18baa5dc8b50c977459820a090", "tokenizer": "b014e8bba4958b120af2d0c1c63eabb7c00379f2bacaf10df7c5325efd2ea467", "vocab/key2row": "76be8b528d0075f7aae98d6fa57a6d3c83ae480a8469e668d7b0af968995ac71", "vocab/lookups.bin": "fce9c883c56165f29573cc938c2a1c9d417ac61bd8f56b671dd5f7996de70682", "vocab/strings.json": "b1966d1f07a05b68576df07f43f40203b4e11124ad82cc839e8312ad8d7fdae7", "vocab/vectors": "14772b683e726436d5948ad3fff2b43d036ef2ebbe3458aafed6004e05a40706", "vocab/vectors.cfg": "ff4359091952c8cd16f1f0482f5770fb82d1707368d5cca3c46aa501f552e3c5"}, "versions": {"en-core-web-sm": "3.8.0", "numpy": "2.1.3", "spacy": "3.8.11", "thinc": "8.3.11"}}'

for name, source in MODULES.items():
    payload=source.encode('utf-8')
    if hashlib.sha256(payload).hexdigest()!=MODULE_HASHES[name]: raise RuntimeError('Module hash differs: '+name)
    (SOURCE/name).write_bytes(payload)
(SOURCE/'trace_cases.json').write_text(PACKET, encoding='utf-8')
(SOURCE/'source_environment.json').write_text(ENVIRONMENT, encoding='utf-8')
(OUTPUT/'trace_cases.json').write_text(PACKET, encoding='utf-8')
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(module_sha256=MODULE_HASHES,
    packet_sha256=hashlib.sha256(PACKET.encode('utf-8')).hexdigest(),
    source_environment_sha256=hashlib.sha256(ENVIRONMENT.encode('utf-8')).hexdigest(),
    original_roi_source_version=126, dataset_required=False, bootstrap_policy='venv_without_pip_base_pip_python'), indent=2), encoding='utf-8')
print('45 small cases; no Dataset/API token/raw features needed. CPU only.', flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete = False
try:
    import numpy as np
    os.environ['OMP_NUM_THREADS'] = '2'
    os.environ['OPENBLAS_NUM_THREADS'] = '2'
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    venv = SOURCE/'pos-env'
    run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', venv], timeout=180)
    PYTHON = venv/'bin/python'
    if '--python' not in subprocess.check_output([sys.executable, '-m', 'pip', '--help'], text=True):
        raise RuntimeError('Base pip must support --python (pip>=22.3); do not install into global environment.')
    PIP = [sys.executable, '-m', 'pip', '--python', str(PYTHON)]
    constraint = SOURCE/'constraints.txt'
    constraint.write_text('numpy==2.1.3'+chr(10), encoding='utf-8')
    run(PIP + ['install', '--disable-pip-version-check', '--only-binary=:all:', '-c', constraint, 'spacy==3.8.11', 'thinc==8.3.11'], timeout=900)
    wheel = SOURCE/'en_core_web_sm-3.8.0-py3-none-any.whl'
    payload = urllib.request.urlopen('https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', timeout=180).read()
    if hashlib.sha256(payload).hexdigest() != '1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85': raise RuntimeError('Official POS model wheel SHA differs')
    wheel.write_bytes(payload)
    run(PIP + ['install', '--no-deps', '--force-reinstall', wheel], timeout=180)
    (OUTPUT/'software_freeze.txt').write_text(subprocess.check_output(PIP + ['freeze', '--all'], text=True), encoding='utf-8')
    run([PYTHON, '-u', SOURCE/'run_lemma_trace.py', '--output', OUTPUT], timeout=900)
    summary=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    complete=summary.get('status')=='LEMMA_TRACE_COMPLETE'
    if not complete: raise RuntimeError('Trace incomplete')
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
except BaseException:
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(), encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete, bank_modified=False, roi_modified=False, dataset_created=False)), encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file(): z.write(p,OUTPUT.name+'/'+p.name)
    print('SEND REVIEW ZIP:',archive,flush=True)
    print('TRACE COMPLETE; original lemmas retained.' if complete else 'INCOMPLETE: send ZIP/log.',flush=True)
